# 頭部與軀幹角度&盒需圖.ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io


In [ ]:
def calculate_head_torso_angle(x_head, y_head, x_shoulder, y_shoulder):
    delta_x = x_head - x_shoulder
    delta_y = y_head - y_shoulder
    angle_radians = math.atan2(delta_y, delta_x)
    return math.degrees(angle_radians)


In [ ]:
category_angles = {}


In [ ]:
for category in range(1, 5):
    print(f"請上傳第 {category} 類的CSV文件")
    uploaded = files.upload()


    for file_name in uploaded.keys():
        df = pd.read_csv(io.BytesIO(uploaded[file_name]))


        kpts_columns = df['kpts'].str.split(',', expand=True)


        for i in range(kpts_columns.shape[1]):
            kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


        if 'kpt_1' in kpts_columns.columns:
            kpts_columns['kpt_1'] = kpts_columns['kpt_1'].str.replace('"', '').astype(float)


        df = df.join(kpts_columns, rsuffix='_new')


        for col in kpts_columns.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')


        x_head = (df['kpt_1'].astype(float) + df['kpt_2'].astype(float)) / 2
        y_head = (df['kpt_3'].astype(float) + df['kpt_4'].astype(float)) / 2

        x_shoulder = (df['kpt_5'].astype(float) + df['kpt_6'].astype(float)) / 2
        y_shoulder = (df['kpt_7'].astype(float) + df['kpt_8'].astype(float)) / 2


        df['head_torso_angle'] = [
            calculate_head_torso_angle(x_head.iloc[i], y_head.iloc[i], x_shoulder.iloc[i], y_shoulder.iloc[i])
            for i in range(len(df))
        ]


        if category not in category_angles:
            category_angles[category] = []
        category_angles[category].extend(df['head_torso_angle'].values)


        print(f"類別 {category} 的頭部與軀幹角度：", df[['head_torso_angle']].head())


In [ ]:
plt.figure(figsize=(10, 6))


In [ ]:
data = [category_angles[cat] for cat in range(1, 5)]
plt.boxplot(data, patch_artist=True, boxprops=dict(facecolor='skyblue'))
plt.title('Comparison of Head to Torso Angles for All Categories', fontsize=16)
plt.xlabel('Category', fontsize=14)
plt.ylabel('Head to Torso Angle (Degrees)', fontsize=14)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot([category_angles[3]], patch_artist=True, boxprops=dict(facecolor='lightblue'), showfliers=True)
plt.title('Category 4 Head to Torso Angle Distribution', fontsize=16)
plt.ylabel('Head to Torso Angle (Degrees)', fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
plt.tight_layout()
plt.show()
